## [HashiCorp Vault](https://developer.hashicorp.com/vault)

ist eine zentrale Plattform für Secret-Management, Identity und Verschlüsselung in modernen Infrastruktur- und Kubernetes-Umgebungen. 

Vault ermöglicht die sichere Verwaltung von Passwörtern, API-Keys, Zertifikaten und dynamischen Credentials und integriert sich direkt mit Kubernetes, Cloud-Plattformen und CI/CD-Systemen. 

Secrets können entweder als Kubernetes Secrets synchronisiert oder erst zur Laufzeit direkt in Pods injiziert werden. Dadurch eignet sich Vault besonders für Plattformen mit hohen Anforderungen an Security, Auditierung, Rotation und Zero-Trust-Architekturen.

Für Kubernetes existieren verschiedene Integrationsmodelle wie Vault Agent Injector, CSI Driver und Operators. Zusätzlich kann Vault mit GitOps-Workflows kombiniert werden.



Installation von Vault 
* im Dev-Mode
* CSI Driver Support
* ohne produktive Persistenz/TLS/HA
* ohne Vault Agent Injector

In [ ]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

helm install vault hashicorp/vault \
  --namespace vault \
  --create-namespace \
  --set "server.dev.enabled=true" \
  --set "server.dev.devRootToken=root" \
  --set "csi.enabled=true" \
  --set "injector.enabled=false"  

Für Vault CSI brauchst du zusätzlich den **Secrets Store CSI Driver**, weil Vault nur den Provider liefert:

In [ ]:
%%bash
helm repo add secrets-store-csi-driver https://kubernetes-sigs.github.io/secrets-store-csi-driver/charts
helm repo update

helm install secrets-store-csi-driver secrets-store-csi-driver/secrets-store-csi-driver \
  --namespace kube-system \
  --set syncSecret.enabled=false

In [ ]:
%%bash
kubectl --namespace=kube-system get pods -l "app=secrets-store-csi-driver"

Vault konfigurieren

In [ ]:
%%bash
kubectl exec -n vault vault-0 -- sh -c '
export VAULT_ADDR=http://127.0.0.1:8200
export VAULT_TOKEN=root

vault auth enable kubernetes || true

vault write auth/kubernetes/config \
  kubernetes_host=https://kubernetes.default.svc

vault kv put secret/app password=supersecret

vault policy write app - <<EOF
path "secret/data/app" {
  capabilities = ["read"]
}
EOF

vault write auth/kubernetes/role/app \
  bound_service_account_names=app \
  bound_service_account_namespaces=default \
  policies=app \
  ttl=1h
'

Dann CSI Resource und Pod

In [ ]:
%%bash
set -euo pipefail

kubectl apply -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  name: app
  namespace: default
---
apiVersion: secrets-store.csi.x-k8s.io/v1
kind: SecretProviderClass
metadata:
  name: vault-app-secret
  namespace: default
spec:
  provider: vault
  parameters:
    vaultAddress: "http://vault.vault:8200"
    roleName: "app"
    objects: |
      - objectName: "db-password"
        secretPath: "secret/data/app"
        secretKey: "password"
---
apiVersion: v1
kind: Pod
metadata:
  name: vault-csi-demo
  namespace: default
spec:
  serviceAccountName: app
  containers:
  - name: app
    image: busybox:1.36
    command: ["sh", "-c", "sleep 3600"]
    volumeMounts:
    - name: vault-secrets
      mountPath: "/mnt/secrets"
      readOnly: true
  volumes:
  - name: vault-secrets
    csi:
      driver: secrets-store.csi.k8s.io
      readOnly: true
      volumeAttributes:
        secretProviderClass: vault-app-secret
EOF

Testen

In [ ]:
%%bash
set -euo pipefail

kubectl wait --for=condition=Ready pod/vault-csi-demo --timeout=120s

kubectl exec vault-csi-demo -- ls -la /mnt/secrets
kubectl exec vault-csi-demo -- cat /mnt/secrets/db-password
kubectl get secrets

---

### Aufräumen


In [ ]:
%%bash
helm uninstall vault -n vault 
kubectl delete ns vault
helm uninstall secrets-store-csi-driver --namespace kube-system

### Links

* [HashiCorp Vault](https://developer.hashicorp.com/vault)
* [OpenBao](https://openbao.org) offener Community-Fork des ursprünglichen Vault-Ansatzes.